# Task 03: Data Ingestion, Cleaning & Preprocessing with Pandas

**Dataset:** `retail_ab_test_10000.csv` (10,000 rows)

This notebook loads the raw CSV, profiles data quality, removes exact duplicate rows, trims whitespace in text fields, handles missing values when present, checks possible outliers, creates a simple derived feature, and exports `clean_dataset.csv`. Potential outliers are flagged for review rather than automatically deleted.

## 1. Import libraries and load data

Place `retail_ab_test_10000.csv` in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('retail_ab_test_10000.csv')
print('Rows:', len(df))
print('Columns:', len(df.columns))
display(df.head())

## 2. Inspect structure and data quality

In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum())
print('\nExact duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)

## 3. Clean duplicates, text fields, and missing values

Exact duplicate rows are removed. Whitespace is trimmed from text columns. Numeric missing values are filled with the column median; categorical missing values are filled with the most frequent value when one exists. The raw dataset currently has no missing values and no exact duplicate rows, so these steps may not change its row count.

In [ ]:
clean_df = df.copy()
clean_df = clean_df.drop_duplicates()

for col in clean_df.select_dtypes(include=['object', 'string']).columns:
    clean_df[col] = clean_df[col].astype('string').str.strip()

for col in clean_df.select_dtypes(include='number').columns:
    if clean_df[col].isna().any():
        clean_df[col] = clean_df[col].fillna(clean_df[col].median())

for col in clean_df.select_dtypes(include=['object', 'string']).columns:
    if clean_df[col].isna().any():
        modes = clean_df[col].mode(dropna=True)
        if not modes.empty:
            clean_df[col] = clean_df[col].fillna(modes.iloc[0])

print('Rows after cleaning:', len(clean_df))
print('Remaining missing cells:', int(clean_df.isna().sum().sum()))
print('Remaining exact duplicates:', int(clean_df.duplicated().sum()))

## 4. Check possible outliers

The IQR method flags possible numeric outliers. These are not automatically removed because unusual values can be valid observations.

In [ ]:
numeric_cols = clean_df.select_dtypes(include='number').columns
outlier_summary = []
for col in numeric_cols:
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((clean_df[col] < lower) | (clean_df[col] > upper)).sum())
    outlier_summary.append({'column': col, 'lower_limit': lower, 'upper_limit': upper, 'possible_outliers': count})
outlier_report = pd.DataFrame(outlier_summary)
display(outlier_report)

## 5. Create a derived feature

`conversion_flag` mirrors the `purchased` indicator in a clearly named analysis-friendly feature. The original column is retained.

In [ ]:
clean_df['conversion_flag'] = clean_df['purchased'].astype(int)
print(clean_df[['purchased', 'conversion_flag']].head())

## 6. Export cleaned dataset

In [ ]:
output_path = 'clean_dataset.csv'
clean_df.to_csv(output_path, index=False)
print(f'Saved {output_path}')
print('Final shape:', clean_df.shape)
display(clean_df.head())

## 7. Final validation

In [ ]:
check_df = pd.read_csv('clean_dataset.csv')
print('Saved file shape:', check_df.shape)
print('Missing cells:', int(check_df.isna().sum().sum()))
print('Duplicate rows:', int(check_df.duplicated().sum()))
print('Columns:', check_df.columns.tolist())